# Ablation Study (leave-one-out)

Isolates the contribution of each ICDN component by disabling one module at a
time and re-running a small Optuna search per variant. Two evaluation axes,
matching the referee's request: predictive performance (R2 / MAE / RMSE) and
elasticity stability + plausibility (elast_score and its cross-fold std).

Variants (leave-one-out): full, no_smooth, no_elast, no_attention, no_cross, no_splines.
No seed loop, no bootstrap: stability is estimated across temporal folds only.
No changes to src/ are required:
  - cross-price  -> existing use_cross=False flag
  - attention    -> uniform-weight monkeypatch of the neighbor selector
  - splines      -> zero + freeze of the spline heads

# Imports

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import json
import types
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import random
import optuna
from pathlib import Path

# -- Personal Libraries
from src.dominick import DominickDataLoader
from src.dominick.multiproduct_builder import MultiProductBuilder
from src.nn.data import ColumnEncoder, DataLoaderFactory, SplineBuilder
from src.nn.spline import MultiCubicSplineBasis
from src.nn.models import IntegrableDemandHead, ICDN
from src.nn.loss import ElasticityLoss
from src.multiproduct import MultiProductDataset, ProductTokenBuilder
from src.utils import TemporalSplitter
from src.nn.heads.neighbor_selector import SparseNeighborSelector  # for the attention monkeypatch

/home/thebigmonster/Github/nn-elasticity/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Settings 

In [3]:
# initial seed
BASE_SEED = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ── Data ──────────────────────────────────────────────────────────
N_UPCS = 5          # Number of UPCs
SMOOTH_WINDOW = 8   # Smoothing window for phase 0
BETA_EDA = -2       # Beta for initialization phase 0
K_NEIGHBORS = 5     # Number of neighbors for the product

# ── Ablation tuning (no seeds, no bootstrap) ──────────────────────
N_TRIALS_ABL = 10   # ~10 trials per variant, as requested
N_FOLDS = 3         # folds are the ONLY variance source here (stability axis)
MIN_TRAIN_FRAC = 0.50

# ── Training ──────────────────────────────────────────────────────
N_EPOCHS_P0 = 250
N_EPOCHS_P1 = 300
PATIENCE    = 20    # epochs before reducing the learning rate
ES_PATIENCE = 40    # epochs before early stopping

# ── Dimensionality for sku-level features ───────────────────────────
D_STORE = 16
D_BRAND = 8
D_STYLE = 8

# ── Checkpoints (own folder to avoid clashing with hparam-search) ──
CKPT_DIR = Path("../results/checkpoints/ablation")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Results ─────────────────────────────────────────────────────
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ABL_DB           = "sqlite:///../results/ablation_studies.db"  # one DB, one study per variant
ABL_RECORDS_PATH = RESULTS_DIR / "ablation_fold_records.csv"
ABL_SUMMARY_PATH = RESULTS_DIR / "ablation_summary.csv"

Device: cuda


# Seeds

In [4]:
# Function to set all seeds and make the results reproducible
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(BASE_SEED)

# Code - Loader

In [5]:
# Load the dataset
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv").copy()
print(f"Dataset shape: {df.shape}")

# Encode the categorical variables
encoder = ColumnEncoder()
_, store_cats = encoder.factorize(df, "store_code", sort=True)
_, week_cats  = encoder.factorize(df, "week_id", sort=True)
_, brand_cats = encoder.factorize(df, "brand_family_norm",  sort=True)
_, style_cats = encoder.factorize(df, "style_segment_norm", sort=True)

n_stores = len(store_cats)
n_weeks  = len(week_cats)
n_brands = len(brand_cats)
n_styles = len(style_cats)
print(f"Stores: {n_stores}  |  Weeks: {n_weeks}  |  Brands: {n_brands}  |  Styles: {n_styles}")

# Map brand/style numeric codes to 1,2,... (0 reserved for unknown/missing)
brand_map = {v: i + 1 for i, v in enumerate(brand_cats)}
style_map = {v: i + 1 for i, v in enumerate(style_cats)}
df["brand_family_norm"]  = df["brand_family_norm"].map(brand_map).fillna(0).astype(int)
df["style_segment_norm"] = df["style_segment_norm"].map(style_map).fillna(0).astype(int)

# Build the multi-product (wide-format) dataset
mp_builder = MultiProductBuilder()
mp_builder.fit(df, n_upcs=N_UPCS)
full_wide_raw = mp_builder.transform().copy()
n_upcs = mp_builder.n

print(f"Full wide shape: {full_wide_raw.shape}")
print(f"UPCs selected: {n_upcs}")
print(f"Top UPCs: {mp_builder.selected_upcs[:N_UPCS]}")

Dataset shape: (463722, 44)
Stores: 70  |  Weeks: 302  |  Brands: 54  |  Styles: 13
Full wide shape: (19808, 171)
UPCs selected: 5
Top UPCs: [3410010505, 7289000011, 1820000784, 8248812345, 3410017306]


# Neighbor meta

In [6]:
# Static metadata per UPC position — used by neighbor-aware attention in the model
upc_meta = (
    df.groupby("upc_code")[["category_code", "brand_family_norm",
                             "style_segment_norm", "liters_per_upc"]]
    .first()
    .loc[mp_builder.selected_upcs]
)

cat_codes, _ = pd.factorize(upc_meta["category_code"], sort=True)

neighbor_meta = {
    "category": torch.tensor(cat_codes, dtype=torch.long, device=device),
    "brand":    torch.tensor(upc_meta["brand_family_norm"].values,   dtype=torch.long,    device=device),
    "style":    torch.tensor(upc_meta["style_segment_norm"].values,  dtype=torch.long,    device=device),
    "liters":   torch.tensor(upc_meta["liters_per_upc"].values,      dtype=torch.float32, device=device),
}
print("neighbor_meta built")

neighbor_meta built


# Temporal Folds

In [7]:
splitter = TemporalSplitter(week_col="week_id")
fold_splits = splitter.expanding_splits(
    df=full_wide_raw,
    n_folds=N_FOLDS,
    min_train_frac=MIN_TRAIN_FRAC,
)

print(f"N folds available: {len(fold_splits)}")
for i, (train_fold, val_fold) in enumerate(fold_splits):
    print(
        f"Fold {i}: train={len(train_fold):,} "
        f"val={len(val_fold):,} "
        f"train_weeks={train_fold['week_id'].nunique()} "
        f"val_weeks={val_fold['week_id'].nunique()}"
    )

N folds available: 3
Fold 0: train=9,756 val=3,394 train_weeks=151 val_weeks=50
Fold 1: train=13,150 val=3,339 train_weeks=201 val_weeks=50
Fold 2: train=16,489 val=3,254 train_weeks=251 val_weeks=50


# Fold-Frame + loader Helpers

In [8]:
# Global store/week code maps (must stay consistent across folds)
store_map = {v: i for i, v in enumerate(store_cats)}
week_map  = {v: i for i, v in enumerate(week_cats)}


# Encodes store/week codes, sorts, and smooths log liters for phase 0.
def build_fold_frames(train_wide, val_wide, smooth_window: int):
    train_wide = train_wide.copy()
    val_wide   = val_wide.copy()

    for w in [train_wide, val_wide]:
        w["store_code"] = w["store_code"].map(store_map)
        w["week_id"]    = w["week_id"].map(week_map)

    train_wide_s = train_wide.sort_values(["store_code", "week_id"]).copy()
    val_wide_s   = val_wide.sort_values(["store_code", "week_id"]).copy()

    # Rolling-average smoothing of the targets, used only for the phase-0 warm start.
    for i in range(n_upcs):
        col = f"log_liters_{i}"
        for df_w in [train_wide_s, val_wide_s]:
            df_w[col] = (
                df_w.groupby("store_code")[col]
                .transform(lambda s: s.rolling(window=smooth_window, min_periods=1).mean())
            )

    return train_wide, val_wide, train_wide_s, val_wide_s


# Builds the phase-0 (smoothed) and phase-1 (raw) train/val DataLoaders.
def build_loaders(train_wide, val_wide, train_wide_s, val_wide_s, batch_size: int):
    loader_factory = DataLoaderFactory(
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
    )

    train_ds_p0 = MultiProductDataset(train_wide_s, n=n_upcs)
    val_ds_p0   = MultiProductDataset(val_wide_s,   n=n_upcs)
    train_ds    = MultiProductDataset(train_wide,   n=n_upcs)
    val_ds      = MultiProductDataset(val_wide,     n=n_upcs)

    train_loader_p0 = loader_factory.create_train_loader(train_ds_p0, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader_p0   = loader_factory.create_eval_loader(val_ds_p0,    batch_size=batch_size, shuffle=False)
    train_loader    = loader_factory.create_train_loader(train_ds,    batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader      = loader_factory.create_eval_loader(val_ds,       batch_size=batch_size, shuffle=False)
    return train_loader_p0, val_loader_p0, train_loader, val_loader

# Freeze - Init Helpers

In [9]:
def freeze_nonlinear(model):
    # Freeze all spline heads (own and cross) and the bilinear interaction head.
    # Reduces the model to a log-linear demand during phase 0.
    for attr in ["head_w", "head_w_cross", "head_cross"]:
        if hasattr(model.head.param_head, attr):
            head = getattr(model.head.param_head, attr)
            head.weight.requires_grad_(False)
            head.bias.requires_grad_(False)

def unfreeze_nonlinear(model):
    # Unfreeze all spline and bilinear heads for phase 1.
    for attr in ["head_w", "head_w_cross", "head_cross"]:
        if hasattr(model.head.param_head, attr):
            head = getattr(model.head.param_head, attr)
            head.weight.requires_grad_(True)
            head.bias.requires_grad_(True)

# Initialize the beta prior so the INITIAL own-price slope equals beta_target
# (~ -2 from EDA) regardless of whether the negative-sign transform is active.
# NOTE: this is only an initialization (a soft nudge), not a constraint: with
# enforce_negative_beta=False the model is free to move beta to any sign.
def init_beta_prior(model, beta_target, enforce_negative_beta=True):
    with torch.no_grad():
        model.head.param_head.head_beta.weight.zero_()
        if enforce_negative_beta:
            # beta = -softplus(beta_raw): invert softplus to recover beta_target.
            beta_raw_init = torch.log(
                torch.exp(torch.tensor(-float(beta_target), dtype=torch.float32)) - 1.0
            )
        else:
            # beta = beta_raw directly: set the bias to beta_target as-is.
            beta_raw_init = torch.tensor(float(beta_target), dtype=torch.float32)
        model.head.param_head.head_beta.bias.fill_(beta_raw_init)

print("Helpers defined")

Helpers defined


# Ablation

In [10]:
# ── Ablation switch 1: true linear model (splines OFF) ─────────────
def zero_and_freeze_splines(model):
    """Set the spline heads to EXACTLY zero and freeze them.
    freeze_nonlinear() only stops gradients but leaves the random init, which
    would still inject a nonzero spline contribution. Zeroing forces
    w = w_cross = u = 0 for every input -> a pure log-linear demand."""
    ph = model.head.param_head
    for attr in ["head_w", "head_w_cross", "head_cross"]:
        if hasattr(ph, attr):
            layer = getattr(ph, attr)
            with torch.no_grad():
                layer.weight.zero_()
                if layer.bias is not None:
                    layer.bias.zero_()
            layer.weight.requires_grad_(False)
            if layer.bias is not None:
                layer.bias.requires_grad_(False)


# ── Ablation switch 2: uniform attention (attention OFF) ───────────
# Keep a handle to the original run so both the online and frozen-graph paths
# are reused; we only overwrite the weights.
_ORIG_SELECTOR_RUN = SparseNeighborSelector.run

def _uniform_selector_run(self, h, category, brand, style, liters):
    """Same neighbor graph as the learned selector, but replace the softmax
    edge weights by uniform 1/k_eff weights. Isolates *learned attention*
    from *having cross-price neighbors at all*."""
    pairs, _ = _ORIG_SELECTOR_RUN(self, h, category, brand, style, liters)
    B, n, _ = h.shape
    E = pairs.shape[1]
    if E == 0:
        return pairs, torch.empty(B, 0, dtype=h.dtype, device=h.device)
    k_eff = E // n
    edge_weights = torch.full((B, E), 1.0 / k_eff, dtype=h.dtype, device=h.device)
    return pairs, edge_weights

def disable_attention(model):
    """Bind the uniform-weight run() onto the selector instance, preserving the
    learned q/k projections that decide the graph structure."""
    sel = model.head.neighbor_selector
    if sel is not None:
        sel.run = types.MethodType(_uniform_selector_run, sel)

print("Ablation switches defined")

Ablation switches defined


# Metrics Helpers

In [11]:
# Hidden options for the encoder (Optuna categorical)
HIDDEN_OPTIONS = {
    "64_32":        (64, 32),
    "128_64":       (128, 64),
    "192_96":       (192, 96),
    "256_128":      (256, 128),
    "256_128_64":   (256, 128, 64),
}

# Global predictive metrics on observed entries only.
def compute_global_metrics(model, val_loader, device):
    model.eval()
    all_true, all_pred = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            y_true   = batch["demands"]
            obs_mask = batch["obs_mask"]
            y_hat, _, _ = model(batch, return_parts=True, neighbor_meta=neighbor_meta)

            mask = obs_mask.bool()
            all_true.append(y_true[mask].cpu())
            all_pred.append(y_hat[mask].cpu())

    y_true_all = torch.cat(all_true).float()
    y_pred_all = torch.cat(all_pred).float()

    err  = y_true_all - y_pred_all
    mae  = float(err.abs().mean())
    rmse = float(torch.sqrt((err ** 2).mean()))

    ss_res = float((err ** 2).sum())
    ss_tot = float(((y_true_all - y_true_all.mean()) ** 2).sum())
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    return {"mae_val": mae, "rmse_val": rmse, "r2_val": r2}


# Elasticity plausibility score in [0, 1]; higher is better.
# Own-price target range [-5, 0], cross-price target range [-1, 1].
def compute_elasticity_score(model, val_loader, device,
                             own_min=-5.0, own_max=0.0,
                             cross_min=-1.0, cross_max=1.0):
    model.eval()
    all_own, all_cross = [], []
    off_diag = ~torch.eye(model.n, dtype=torch.bool, device=device).unsqueeze(0)  # (1, n, n)

    with torch.no_grad():
        for batch in val_loader:
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            obs_mask = batch["obs_mask"].bool()
            _, eps_hat, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)

            E = aux["E"]
            all_own.append(eps_hat[obs_mask].cpu())
            pair_mask  = obs_mask.unsqueeze(2) & obs_mask.unsqueeze(1)
            cross_mask = pair_mask & off_diag
            all_cross.append(E[cross_mask].cpu())

    own   = torch.cat(all_own).numpy()
    cross = torch.cat(all_cross).numpy() if all_cross else np.array([])

    # ── Own score: fraction in range, penalised by deviation from the EDA prior ──
    own_in_range  = float(((own >= own_min) & (own <= own_max)).mean())
    median_own    = float(np.median(own))
    deviation     = max(0.0, abs(median_own - BETA_EDA) - 0.3)
    prior_penalty = min(deviation / abs(BETA_EDA), 1.0)
    own_score     = own_in_range * (1.0 - prior_penalty)

    # ── Cross score: fraction in range ──
    if len(cross) > 0:
        cross_in_range = float(((cross >= cross_min) & (cross <= cross_max)).mean())
        median_cross   = float(np.median(cross))
    else:
        cross_in_range = 1.0
        median_cross   = float("nan")

    score = 0.7 * own_score + 0.3 * cross_in_range

    return {
        "elast_score":             float(score),
        "own_score":               float(own_score),
        "own_in_range":            float(own_in_range),
        "own_elasticity_median":   median_own,
        "cross_in_range":          float(cross_in_range),
        "cross_elasticity_median": median_cross,
    }

# Descriptive, target-free plausibility report. Unlike compute_elasticity_score
# (whose ranges coincide with the training penalty and the selection criterion),
# these are raw distributional facts. The most honest signal is own_frac_negative
# under the unconstrained variant: sign that survives WITHOUT being imposed.
def compute_plausibility_report(model, val_loader, device):
    model.eval()
    all_own, all_cross = [], []
    off_diag = ~torch.eye(model.n, dtype=torch.bool, device=device).unsqueeze(0)

    with torch.no_grad():
        for batch in val_loader:
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            obs_mask = batch["obs_mask"].bool()
            _, eps_hat, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
            E = aux["E"]
            all_own.append(eps_hat[obs_mask].cpu())
            pair_mask = obs_mask.unsqueeze(2) & obs_mask.unsqueeze(1)
            all_cross.append(E[(pair_mask & off_diag)].cpu())

    own   = torch.cat(all_own).numpy()
    cross = torch.cat(all_cross).numpy() if all_cross else np.array([])

    rep = {
        "own_frac_negative":  float((own < 0).mean()),           # sign (weakly targeted only)
        "own_median":         float(np.median(own)),
        "own_p10":            float(np.percentile(own, 10)),
        "own_p90":            float(np.percentile(own, 90)),
        "own_frac_in_m5_0":   float(((own >= -5)  & (own <= 0)).mean()),
        "own_frac_in_m10_0":  float(((own >= -10) & (own <= 0)).mean()),  # wider, neutral ref
    }
    if len(cross) > 0:
        rep.update({
            "cross_median":        float(np.median(cross)),
            "cross_frac_negative": float((cross < 0).mean()),
            "cross_frac_in_m1_1":  float(((cross >= -1) & (cross <= 1)).mean()),
            "cross_abs_median":    float(np.median(np.abs(cross))),
        })
    else:
        rep.update({"cross_median": float("nan"), "cross_frac_negative": float("nan"),
                    "cross_frac_in_m1_1": float("nan"), "cross_abs_median": float("nan")})
    return rep

print("Plausibility report defined")

print("Metric helpers defined")

Plausibility report defined
Metric helpers defined


# Training Loop

In [12]:
# Two-phase (warm-start) training loop with AMP, grad clipping and early stopping.
def run_training(model, train_loader, val_loader, loss_fn,
                 optimizer, scheduler, n_epochs, es_patience,
                 ckpt_path, device, neighbor_meta, phase_name="",
                 verbose=False):

    best_val_loss = float("inf")
    no_improve    = 0
    scaler        = torch.amp.GradScaler("cuda") if device == "cuda" else None

    for epoch in range(n_epochs):
        # ── Train ──────────────────────────────────────────────────
        model.train()
        total_loss, total_denom = 0.0, 0.0

        for batch in train_loader:
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            y_true   = batch["demands"]
            obs_mask = batch["obs_mask"]

            optimizer.zero_grad()
            if scaler:
                with torch.amp.autocast("cuda"):
                    y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                    loss, logs = loss_fn(y_hat, y_true, obs_mask,
                                         aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                         aux["pairs"], E=aux.get("E"))
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                loss, logs = loss_fn(y_hat, y_true, obs_mask,
                                     aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                     aux["pairs"], E=aux.get("E"))
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            denom        = obs_mask.sum().item()
            total_loss  += logs["loss"].item() * denom
            total_denom += denom

        # ── Val ────────────────────────────────────────────────────
        model.eval()
        val_loss_sum, val_denom = 0.0, 0.0
        with torch.no_grad():
            for batch in val_loader:
                batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
                y_true   = batch["demands"]
                obs_mask = batch["obs_mask"]

                y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                _, logs = loss_fn(y_hat, y_true, obs_mask,
                                  aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                  aux["pairs"], E=aux.get("E"))

                denom        = obs_mask.sum().item()
                val_loss_sum += logs["loss"].item() * denom
                val_denom    += denom

        val_loss = val_loss_sum / max(val_denom, 1.0)
        prev_lr  = optimizer.param_groups[0]["lr"]
        scheduler.step(val_loss)
        new_lr   = optimizer.param_groups[0]["lr"]
        if new_lr < prev_lr:
            no_improve = 0

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve    = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_improve += 1

        if verbose and ((epoch + 1) % 50 == 0 or no_improve == 0):
            print(f"  [{phase_name}] Epoch {epoch+1}  val={val_loss:.4f}")

        if no_improve >= es_patience:
            if verbose:
                print(f"  [{phase_name}] Early stopping at epoch {epoch+1}")
            break

    return best_val_loss

print("run_training defined")

run_training defined


# Build and Train

In [13]:
# Builds and trains one model for a given variant / fold, returns its metrics.
def build_and_train(params, variant_cfg, train_fold, val_fold, fold_id, seed, trial_id=0):
    set_all_seeds(seed)

    use_cross     = variant_cfg["use_cross"]
    use_attention = variant_cfg["use_attention"]
    use_splines   = variant_cfg["use_splines"]
    tag           = variant_cfg["name"]

    # ── Economic-constraint knobs (defaults reproduce the original model) ──
    enforce_neg = variant_cfg.get("enforce_negative_beta", True)  # hard sign constraint
    l_own   = variant_cfg.get("l_own",   -5.0)                    # elasticity penalty bounds
    r_own   = variant_cfg.get("r_own",    0.0)
    l_cross = variant_cfg.get("l_cross", -1.0)
    r_cross = variant_cfg.get("r_cross",  1.0)

    # Build the fold dataframes (raw + smoothed) and the DataLoaders.
    train_wide, val_wide, train_wide_s, val_wide_s = build_fold_frames(
        train_wide=train_fold, val_wide=val_fold, smooth_window=SMOOTH_WINDOW,
    )
    train_loader_p0, val_loader_p0, train_loader, val_loader = build_loaders(
        train_wide, val_wide, train_wide_s, val_wide_s, batch_size=params["BATCH_SIZE"]
    )

    # Unpack hyperparameters.
    n_knots       = params["N_KNOTS"]
    hidden        = HIDDEN_OPTIONS[params["HIDDEN_KEY"]]
    dropout       = params["DROPOUT"]
    act           = params.get("ACT", "gelu")
    lr_p0         = params["LR_P0"]
    lr_p1         = params["LR_P1"]
    lambda_smooth = params["LAMBDA_SMOOTH"]
    lambda_elast  = params["LAMBDA_ELAST"]

    ckpt_p0 = CKPT_DIR / f"{tag}_trial{trial_id}_fold{fold_id}_seed{seed}_p0.pt"
    ckpt_p1 = CKPT_DIR / f"{tag}_trial{trial_id}_fold{fold_id}_seed{seed}_p1.pt"

    # ── Build the spline basis (needed even for no_splines to construct the model) ──
    builder = SplineBuilder()
    spline_configs = []
    for i in range(n_upcs):
        x_i = train_wide[f"log_price_{i}"].values
        config = builder.build_from_data(x_i, n_knots=n_knots, q_min=0.05, q_max=0.95)
        spline_configs.append(config)
    knots = torch.stack([cfg["knots"] for cfg in spline_configs], dim=0)
    shift = torch.tensor([cfg["mean"] for cfg in spline_configs])
    scale = torch.tensor([cfg["std"]  for cfg in spline_configs])
    price_splines = MultiCubicSplineBasis(knots=knots, shift=shift, scale=scale)

    # ── Context token builder ──
    token_builder = ProductTokenBuilder(
        n=n_upcs,
        n_stores=n_stores, d_store=D_STORE,
        n_brands=n_brands, d_brand=D_BRAND,
        n_styles=n_styles, d_style=D_STYLE,
    )

    # ── make_model threads the variant flags (use_cross + use_attention) ──
    def make_model():
        head = IntegrableDemandHead(
            context_dim=token_builder.d_token,
            K_splines=n_knots,
            n=n_upcs,
            k_neighbors=K_NEIGHBORS,
            hidden=hidden,
            act=act,
            dropout=dropout,
            use_cross=use_cross,
            enforce_negative_beta=enforce_neg,
        )
        model = ICDN(
            context_builder=token_builder,
            price_splines=price_splines,
            head=head,
            n=n_upcs,
        ).to(device)
        if use_cross and (not use_attention):
            disable_attention(model)   # uniform edge weights, same graph
        return model

    # ── PHASE 0: log-linear warm start on smoothed targets ─────────────
    m0 = make_model()
    freeze_nonlinear(m0)
    if not use_splines:
        zero_and_freeze_splines(m0)    # true linear model: splines pinned at 0
    init_beta_prior(m0, BETA_EDA, enforce_negative_beta=enforce_neg)
    if use_cross:                      # cross heads only exist when use_cross=True
        with torch.no_grad():
            m0.head.param_head.head_beta_cross.weight.zero_()
            m0.head.param_head.head_beta_cross.bias.zero_()

    loss_p0 = ElasticityLoss(
        huber_delta=1.0, lambda_smooth=lambda_smooth, lambda_elast=lambda_elast,
        l_own=l_own, r_own=r_own, l_cross=l_cross, r_cross=r_cross,  
        reduction="mean",
    )

    decay, no_decay = [], []
    for name, p in m0.named_parameters():
        if not p.requires_grad:
            continue
        if (("head_w" in name) or ("head_cross" in name)
                or ("head_beta_cross" in name) or name.endswith("bias")):
            no_decay.append(p)
        else:
            decay.append(p)
    opt_p0 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr_p0,
    )
    sch_p0 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p0, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )
    run_training(m0, train_loader_p0, val_loader_p0, loss_p0,
                 opt_p0, sch_p0, N_EPOCHS_P0, ES_PATIENCE,
                 ckpt_p0, device, neighbor_meta, "P0")

    # ── PHASE 1: unlock splines on raw targets (unless splines are ablated) ──
    m1 = make_model()
    m1.load_state_dict(torch.load(ckpt_p0, map_location=device))
    if use_splines:
        unfreeze_nonlinear(m1)
    else:
        zero_and_freeze_splines(m1)    # keep splines OFF in phase 1

    loss_p1 = ElasticityLoss(
        huber_delta=1.0, lambda_smooth=lambda_smooth, lambda_elast=lambda_elast,
        l_own=l_own, r_own=r_own, l_cross=l_cross, r_cross=r_cross,  # <── variant bounds
        reduction="mean",
    )
    decay, no_decay = [], []
    for name, p in m1.named_parameters():
        if not p.requires_grad:
            continue
        if (("head_w" in name) or ("head_cross" in name)
                or ("head_beta_cross" in name) or name.endswith("bias")):
            no_decay.append(p)
        else:
            decay.append(p)
    opt_p1 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr_p1,
    )
    sch_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p1, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )
    run_training(m1, train_loader, val_loader, loss_p1,
                 opt_p1, sch_p1, N_EPOCHS_P1, ES_PATIENCE,
                 ckpt_p1, device, neighbor_meta, "P1")
    m1.load_state_dict(torch.load(ckpt_p1, map_location=device))

    # ── Freeze the sparse graph (guarded: selector is None when use_cross=False) ──
    m1.eval()
    selector = m1.head.neighbor_selector
    if selector is not None:
        def h_iter(loader):
            with torch.no_grad():
                for batch in loader:
                    batch  = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
                    tokens = m1.context_builder(batch)
                    h      = m1.head.encoder(tokens)
                    yield h
        global_mean = selector.accumulate_mean_scores(
            h_iter(train_loader),
            category=neighbor_meta["category"], brand=neighbor_meta["brand"],
            style=neighbor_meta["style"], liters=neighbor_meta["liters"],
        )
        selector.freeze_graph(
            global_mean,
            category=neighbor_meta["category"], brand=neighbor_meta["brand"],
            style=neighbor_meta["style"], liters=neighbor_meta["liters"],
        )

    # ── Evaluate ──
    pred_metrics  = compute_global_metrics(m1, val_loader, device)
    elast_metrics = compute_elasticity_score(m1, val_loader, device)
    plaus_report  = compute_plausibility_report(m1, val_loader, device)   # <── raw, target-free
    out = {
        "variant": tag, "trial_id": trial_id, "fold": fold_id, "seed": seed,
        "n_train": len(train_wide), "n_val": len(val_wide),
        "enforce_negative_beta": enforce_neg,
        "l_own": l_own, "r_own": r_own, "l_cross": l_cross, "r_cross": r_cross,
        **pred_metrics, **elast_metrics, **plaus_report,
    }

    print(
        f"[{tag}] trial={trial_id} fold={fold_id} | "
        f"R2={out['r2_val']:.4f} MAE={out['mae_val']:.4f} | "
        f"ElastScore={out['elast_score']:.4f} | "
        f"own[pct={100*out['own_in_range']:.1f}% med={out['own_elasticity_median']:.2f}] "
        f"cross[pct={100*out['cross_in_range']:.1f}% med={out['cross_elasticity_median']:.2f}]"
    )

    ckpt_p0.unlink(missing_ok=True)
    ckpt_p1.unlink(missing_ok=True)
    return out

print("build_and_train defined")

build_and_train defined


# Variant table

In [14]:
# Leave-one-out variants. fix_* pins a hyperparameter that is inert for the
# variant, so Optuna does not waste trials searching an irrelevant dimension.
VARIANTS = {
    # ── Family 1: module ablation (referee critique on component contribution) ──
    "full":         dict(use_cross=True,  use_attention=True,  use_splines=True),
    "no_smooth":    dict(use_cross=True,  use_attention=True,  use_splines=True,  fix_lambda_smooth=0.0),
    "no_elast":     dict(use_cross=True,  use_attention=True,  use_splines=True,  fix_lambda_elast=0.0),
    "no_attention": dict(use_cross=True,  use_attention=False, use_splines=True),
    "no_cross":     dict(use_cross=False, use_attention=True,  use_splines=True),
    "no_splines":   dict(use_cross=True,  use_attention=True,  use_splines=False,
                         fix_lambda_smooth=0.0, fix_n_knots=2),

    # ── Family 2: economic-constraint sensitivity (referee critique #4) ──
    # Sign constraint OFF, penalty kept: isolates the hard negativity constraint.
    "free_sign":    dict(use_cross=True,  use_attention=True,  use_splines=True,
                         enforce_negative_beta=False, select_by="robust_r2"),

    # Sign OFF and elasticity penalty OFF: does plausibility EMERGE from data?
    # Selected by predictive score only -> no circularity with the [-5,0]/[-1,1] ranges.
    "unconstrained":dict(use_cross=True,  use_attention=True,  use_splines=True,
                         enforce_negative_beta=False, fix_lambda_elast=0.0,
                         select_by="robust_r2"),

    # Range sensitivity: penalty ON but with much wider, near-inert bounds.
    "wide_bounds":  dict(use_cross=True,  use_attention=True,  use_splines=True,
                         l_own=-10.0, r_own=2.0, l_cross=-3.0, r_cross=3.0,
                         select_by="robust_r2"),
}
for name, cfg in VARIANTS.items():
    cfg["name"] = name

PARAM_COLS = ["N_KNOTS", "HIDDEN_KEY", "DROPOUT", "LR_P0", "LR_P1",
              "LAMBDA_SMOOTH", "LAMBDA_ELAST", "BATCH_SIZE"]
print("Variants:", list(VARIANTS.keys()))

Variants: ['full', 'no_smooth', 'no_elast', 'no_attention', 'no_cross', 'no_splines', 'free_sign', 'unconstrained', 'wide_bounds']


# Optuna 

In [15]:
def make_objective(variant_cfg, trial_records):
    def objective(trial):
        params = {
            "N_KNOTS":       (variant_cfg["fix_n_knots"] if "fix_n_knots" in variant_cfg
                              else trial.suggest_int("N_KNOTS", 2, 16)),
            "HIDDEN_KEY":    trial.suggest_categorical("HIDDEN_KEY", list(HIDDEN_OPTIONS.keys())),
            "DROPOUT":       trial.suggest_float("DROPOUT", 0.0, 0.3),
            "LR_P0":         trial.suggest_float("LR_P0", 1e-4, 1e-2, log=True),
            "LR_P1":         trial.suggest_float("LR_P1", 1e-5, 5e-3, log=True),
            "LAMBDA_SMOOTH": (variant_cfg["fix_lambda_smooth"] if "fix_lambda_smooth" in variant_cfg
                              else trial.suggest_float("LAMBDA_SMOOTH", 1e-5, 0.2, log=True)),
            "LAMBDA_ELAST":  (variant_cfg["fix_lambda_elast"] if "fix_lambda_elast" in variant_cfg
                              else trial.suggest_float("LAMBDA_ELAST", 1e-5, 0.2, log=True)),
            "BATCH_SIZE":    trial.suggest_categorical("BATCH_SIZE", [256, 512, 1024]),
        }

        print(f"\n{'='*70}\n[{variant_cfg['name']}] Trial {trial.number}")
        for k, v in params.items():
            print(f"  {k}: {v}")
        print(f"{'='*70}")

        # One run per fold, single fixed seed (no seed loop, no bootstrap).
        run_rows = []
        for fold_id, (train_fold, val_fold) in enumerate(fold_splits):
            run_rows.append(build_and_train(
                params=params, variant_cfg=variant_cfg,
                train_fold=train_fold, val_fold=val_fold,
                fold_id=fold_id, seed=BASE_SEED, trial_id=trial.number,
            ))

        df_trial   = pd.DataFrame(run_rows)
        mean_r2    = float(df_trial["r2_val"].mean())
        std_r2     = float(df_trial["r2_val"].std(ddof=1)) if len(df_trial) > 1 else 0.0
        mean_elast = float(df_trial["elast_score"].mean())
        std_elast  = float(df_trial["elast_score"].std(ddof=1)) if len(df_trial) > 1 else 0.0
        mean_mae   = float(df_trial["mae_val"].mean())
        mean_rmse  = float(df_trial["rmse_val"].mean())

        # Robust scores penalise cross-fold variance (stability).
        robust_r2    = mean_r2    - 0.25 * std_r2
        robust_elast = mean_elast - 0.25 * std_elast

        for k, v in {
            "mean_r2": mean_r2, "std_r2": std_r2,
            "mean_elast_score": mean_elast, "std_elast_score": std_elast,
            "mean_mae": mean_mae, "mean_rmse": mean_rmse,
            "robust_r2": robust_r2, "robust_elast": robust_elast,
        }.items():
            trial.set_user_attr(k, v)

        df_trial["trial"] = trial.number
        for k, v in params.items():
            df_trial[k] = v
        trial_records.extend(df_trial.to_dict(orient="records"))

        print(
            f"[{variant_cfg['name']}] Trial {trial.number} | "
            f"mean_R2={mean_r2:.4f} std_R2={std_r2:.4f} | "
            f"mean_Elast={mean_elast:.4f} std_Elast={std_elast:.4f}"
        )
        return robust_r2, robust_elast
    return objective

print("Objective factory defined")

Objective factory defined


# Study

In [16]:
# WARNING: this is the heavy cell. ~6 variants x 10 trials x 3 folds two-phase
# trainings. Optuna's sqlite storage makes it resumable (load_if_exists=True).
all_records      = []
best_per_variant = []

for name, cfg in VARIANTS.items():
    print(f"\n{'#'*70}\n# VARIANT: {name}\n{'#'*70}")
    trial_records = []

    study = optuna.create_study(
        directions=["maximize", "maximize"],
        study_name=f"ablation_{name}",
        storage=ABL_DB,
        load_if_exists=True,
    )
    study.optimize(make_objective(cfg, trial_records), n_trials=N_TRIALS_ABL)

    df_v = pd.DataFrame(trial_records)
    all_records.append(df_v)

    # Aggregate per trial and pick the best by combined robust score.
    per_trial = (
        df_v.groupby("trial")
            .agg(mean_r2=("r2_val", "mean"), std_r2=("r2_val", "std"),
                mean_elast=("elast_score", "mean"), std_elast=("elast_score", "std"),
                mean_mae=("mae_val", "mean"),
                own_frac_negative=("own_frac_negative", "mean"),
                own_median=("own_median", "mean"),
                own_frac_in_m5_0=("own_frac_in_m5_0", "mean"),
                cross_frac_in_m1_1=("cross_frac_in_m1_1", "mean"))
            .fillna(0.0)
    )
    per_trial["robust_r2"]    = per_trial["mean_r2"] - 0.25 * per_trial["std_r2"]
    per_trial["robust_score"] = (
        per_trial["robust_r2"]
        + (per_trial["mean_elast"] - 0.25 * per_trial["std_elast"])
    )
    select_by = cfg.get("select_by", "robust_score")
    best_trial_id = int(per_trial[select_by].idxmax())
    best = per_trial.loc[best_trial_id]

    # Recover the winning hyperparameters (identical across a trial's fold rows).
    best_params = df_v[df_v["trial"] == best_trial_id].iloc[0][PARAM_COLS].to_dict()
    with open(RESULTS_DIR / f"ablation_best_params__{name}.json", "w") as f:
        json.dump({"variant": name, "best_trial": best_trial_id,
                   "params": {k: (int(v) if k in ("N_KNOTS", "BATCH_SIZE") else v)
                              for k, v in best_params.items()}}, f, indent=2)

    best_per_variant.append({
        "variant": name, "best_trial": best_trial_id, "select_by": select_by,
        "mean_r2": float(best["mean_r2"]), "std_r2": float(best["std_r2"]),
        "mean_elast_score": float(best["mean_elast"]),
        "own_frac_negative": float(best["own_frac_negative"]),   # <── key OOS evidence
        "own_median": float(best["own_median"]),
        "own_frac_in_m5_0": float(best["own_frac_in_m5_0"]),
        "cross_frac_in_m1_1": float(best["cross_frac_in_m1_1"]),
    })

# Persist all per-fold records for later paired analysis.
pd.concat(all_records, ignore_index=True).to_csv(ABL_RECORDS_PATH, index=False)
print(f"\nSaved per-fold records to {ABL_RECORDS_PATH}")


######################################################################
# VARIANT: full
######################################################################


[I 2026-07-19 08:51:31,227] A new study created in RDB with name: ablation_full



[full] Trial 0
  N_KNOTS: 13
  HIDDEN_KEY: 128_64
  DROPOUT: 0.18383493498473885
  LR_P0: 0.0028994749182752544
  LR_P1: 0.00027261614352616284
  LAMBDA_SMOOTH: 1.640100506615793e-05
  LAMBDA_ELAST: 3.5959443068533196e-05
  BATCH_SIZE: 1024
[full] trial=0 fold=0 | R2=0.7177 MAE=0.5255 | ElastScore=0.6043 | own[pct=78.4% med=-0.98] cross[pct=84.5% med=0.45]
[full] trial=0 fold=1 | R2=0.7066 MAE=0.4642 | ElastScore=0.4951 | own[pct=66.5% med=-0.90] cross[pct=72.3% med=0.57]
[full] trial=0 fold=2 | R2=0.5383 MAE=0.4614 | ElastScore=0.8045 | own[pct=88.8% med=-1.47] cross[pct=84.6% med=0.52]


[I 2026-07-19 08:55:04,143] Trial 0 finished with values: [0.6290590269460263, 0.5954186725344945] and parameters: {'N_KNOTS': 13, 'HIDDEN_KEY': '128_64', 'DROPOUT': 0.18383493498473885, 'LR_P0': 0.0028994749182752544, 'LR_P1': 0.00027261614352616284, 'LAMBDA_SMOOTH': 1.640100506615793e-05, 'LAMBDA_ELAST': 3.5959443068533196e-05, 'BATCH_SIZE': 1024}.


[full] Trial 0 | mean_R2=0.6542 std_R2=0.1005 | mean_Elast=0.6346 std_Elast=0.1569

[full] Trial 1
  N_KNOTS: 13
  HIDDEN_KEY: 256_128_64
  DROPOUT: 0.030103152753039486
  LR_P0: 0.00016202686086885172
  LR_P1: 0.002849044279993746
  LAMBDA_SMOOTH: 0.008258039746556202
  LAMBDA_ELAST: 0.02596317337764414
  BATCH_SIZE: 256
[full] trial=1 fold=0 | R2=0.7186 MAE=0.5282 | ElastScore=0.9583 | own[pct=100.0% med=-1.68] cross[pct=88.9% med=0.46]
[full] trial=1 fold=1 | R2=0.6887 MAE=0.4828 | ElastScore=0.9651 | own[pct=100.0% med=-2.17] cross[pct=88.4% med=0.35]
[full] trial=1 fold=2 | R2=0.4810 MAE=0.4946 | ElastScore=0.9379 | own[pct=100.0% med=-1.71] cross[pct=79.3% med=0.41]


...

[wide_bounds] Trial 7 | mean_R2=0.6722 std_R2=0.1311 | mean_Elast=0.7095 std_Elast=0.0617

[wide_bounds] Trial 8
  N_KNOTS: 9
  HIDDEN_KEY: 256_128_64
  DROPOUT: 0.04331253441239001
  LR_P0: 0.0008132857016309759
  LR_P1: 0.00010714157111220989
  LAMBDA_SMOOTH: 0.0010054434011651924
  LAMBDA_ELAST: 2.1320362384986174e-05
  BATCH_SIZE: 1024
[wide_bounds] trial=8 fold=0 | R2=0.7380 MAE=0.5081 | ElastScore=0.7604 | own[pct=100.0% med=-1.17] cross[pct=81.9% med=0.17]
[wide_bounds] trial=8 fold=1 | R2=0.6373 MAE=0.5328 | ElastScore=0.8440 | own[pct=100.0% med=-1.40] cross[pct=83.1% med=0.11]
[wide_bounds] trial=8 fold=2 | R2=0.5327 MAE=0.4709 | ElastScore=0.8732 | own[pct=100.0% med=-1.42] cross[pct=90.3% med=0.42]


[I 2026-07-19 16:39:52,390] Trial 8 finished with values: [0.6103114016130481, 0.8112424862646899] and parameters: {'N_KNOTS': 9, 'HIDDEN_KEY': '256_128_64', 'DROPOUT': 0.04331253441239001, 'LR_P0': 0.0008132857016309759, 'LR_P1': 0.00010714157111220989, 'LAMBDA_SMOOTH': 0.0010054434011651924, 'LAMBDA_ELAST': 2.1320362384986174e-05, 'BATCH_SIZE': 1024}.


[wide_bounds] Trial 8 | mean_R2=0.6360 std_R2=0.1027 | mean_Elast=0.8259 std_Elast=0.0586

[wide_bounds] Trial 9
  N_KNOTS: 14
  HIDDEN_KEY: 256_128
  DROPOUT: 0.26073705972016903
  LR_P0: 0.0004861761398017025
  LR_P1: 0.00040483399247028066
  LAMBDA_SMOOTH: 5.915206566389164e-05
  LAMBDA_ELAST: 0.00019108690621150049
  BATCH_SIZE: 256
[wide_bounds] trial=9 fold=0 | R2=0.7386 MAE=0.5045 | ElastScore=0.6723 | own[pct=94.4% med=-0.93] cross[pct=88.5% med=0.59]
[wide_bounds] trial=9 fold=1 | R2=0.7074 MAE=0.4729 | ElastScore=0.4743 | own[pct=79.3% med=-0.59] cross[pct=75.8% med=0.50]
[wide_bounds] trial=9 fold=2 | R2=0.5455 MAE=0.4555 | ElastScore=0.7938 | own[pct=94.5% med=-1.40] cross[pct=77.1% med=0.56]


[I 2026-07-19 16:47:41,260] Trial 9 finished with values: [0.6379391747187254, 0.6064902636672126] and parameters: {'N_KNOTS': 14, 'HIDDEN_KEY': '256_128', 'DROPOUT': 0.26073705972016903, 'LR_P0': 0.0004861761398017025, 'LR_P1': 0.00040483399247028066, 'LAMBDA_SMOOTH': 5.915206566389164e-05, 'LAMBDA_ELAST': 0.00019108690621150049, 'BATCH_SIZE': 256}.


[wide_bounds] Trial 9 | mean_R2=0.6639 std_R2=0.1037 | mean_Elast=0.6468 std_Elast=0.1612

Saved per-fold records to ../results/ablation_fold_records.csv


# Comparison table

In [17]:
# Final ablation table with deltas vs the full model (both axes).
summary = pd.DataFrame(best_per_variant).set_index("variant").reindex(list(VARIANTS.keys()))

full_row = summary.loc["full"]
summary["d_r2_vs_full"]    = summary["mean_r2"]          - full_row["mean_r2"]
summary["d_elast_vs_full"] = summary["mean_elast_score"] - full_row["mean_elast_score"]

summary.to_csv(ABL_SUMMARY_PATH)
print(summary.round(4).to_string())
print(f"\nSaved summary to {ABL_SUMMARY_PATH}")

               best_trial     select_by  mean_r2  std_r2  mean_elast_score  own_frac_negative  own_median  own_frac_in_m5_0  cross_frac_in_m1_1  d_r2_vs_full  d_elast_vs_full
variant                                                                                                                                                                       
full                    1  robust_score   0.6294  0.1294            0.9538             1.0000     -1.8527            1.0000              0.8551        0.0000           0.0000
no_smooth               9  robust_score   0.6567  0.0937            0.6703             0.9309     -1.8800            0.7126              0.7249        0.0273          -0.2835
no_elast                1  robust_score   0.6278  0.0897            0.8526             1.0000     -1.5779            1.0000              0.6942       -0.0016          -0.1011
no_attention            8  robust_score   0.6332  0.1156            0.9090             1.0000     -1.7780            1.0000  